# When Does Adaptive TCT Beat Standard IDP?

**Standard IDP** charges every active user ε on every round, regardless of whether they contributed to the result.  
**Adaptive TCT** charges only users who contributed to a target hit — plus a one-time registration cost K₁·ε.

TCT wins as long as its registration cost is affordable:

> TCT wins when `K₁·ε < EPS_BUDGET`  ⟺  `K₁ < EPS_BUDGET / ε`

For our parameters (EPS_BUDGET=1.0, ε=0.01) this means `K₁ < 100`. All practical δ values satisfy this.  
The interesting question is *how much* TCT wins, and how that advantage depends on δ and the empirical hit rate.

| Parameter | Effect on TCT | Effect on Standard IDP |
|---|---|---|
| **δ** (larger) | Smaller K₁ → cheaper registration → larger advantage | No effect |
| **hit rate** (lower) | Fewer incremental charges → slightly better | No effect (charges all users regardless of tops) |

**Sections:**
1. Break-even table — registration cost vs. budget
2. δ sweep — how K₁ determines the advantage magnitude
3. Population activity sweep — confirmation that IDP ignores E[rate]
4. Phase diagram — (δ, E[rate]) parameter space
5. Winning regime — full simulation in a favorable setting
6. Hit rate sweep — effect of threshold level

In [ ]:
import sys
sys.path.insert(0, '..')

import math
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

from simulation.population import User
from simulation.accountants.adaptive_tct import AdaptiveTCTAccountant
from simulation.accountants.standard_dp import StandardDPAccountant
from simulation.calibration import q_from_epsilon, set_threshold_for_hit_rate
from simulation.datasets import generate_synthetic_population, PopulationConfig
from simulation.query import compute_round_conversions, laplace_threshold_query

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

In [ ]:
# ── Shared parameters ─────────────────────────────────────────────────────────
EPSILON         = 0.01
N_USERS         = 300
N_ROUNDS        = 300
EPS_BUDGET      = 1.0
TARGET_HIT_RATE = 0.05
P_PARTICIPATE   = 0.3      # fraction of population sampled per TCT and non-TCT query
SEED            = 42

q = q_from_epsilon(EPSILON)
# Standard IDP: each user participates with prob P_PARTICIPATE per round → expected rounds to exhaust
idp_halflife_approx = int(EPS_BUDGET / (P_PARTICIPATE * EPSILON))
print(f'ε={EPSILON}, q={q:.4f}, N_ROUNDS={N_ROUNDS}, EPS_BUDGET={EPS_BUDGET}, P_PARTICIPATE={P_PARTICIPATE}')
print(f'Standard IDP expected exhaustion ≈ {idp_halflife_approx} rounds '
      f'(EPS_BUDGET / (P_PARTICIPATE × ε) = {EPS_BUDGET}/{P_PARTICIPATE*EPSILON:.4f})')


def half_life(active_frac, threshold=0.5):
    for r, f in enumerate(active_frac):
        if f <= threshold:
            return r
    return len(active_frac)


def calibrate_threshold(pop_cfg, p_participate=P_PARTICIPATE, seed=SEED):
    """Calibrate threshold T for target hit rate, accounting for partial participation."""
    rng = np.random.default_rng(seed)
    calib = generate_synthetic_population(pop_cfg, rng=np.random.default_rng(seed))
    n_pilot = 2000
    noisy_counts = np.empty(n_pilot)
    for i in range(n_pilot):
        active = calib.active_users
        participants = [u for u in active if rng.random() < p_participate]
        if not participants:
            noisy_counts[i] = rng.laplace(0.0, 1.0 / EPSILON)
            continue
        true_count, _ = compute_round_conversions(participants, rng)
        noisy_counts[i] = true_count + rng.laplace(0.0, 1.0 / EPSILON)
    return float(np.quantile(noisy_counts, 1.0 - TARGET_HIT_RATE))


def run_tct_only(accountant_factory, pop_cfg, threshold, p_participate=P_PARTICIPATE, seed=SEED):
    """TCT rounds only — no non-TCT queries.
    Each round a random P_PARTICIPATE fraction of active users is sampled;
    only sampled users contribute to the query and are charged.
    """
    rng = np.random.default_rng(seed)
    pop = generate_synthetic_population(pop_cfg, rng=np.random.default_rng(seed))
    acc = accountant_factory()
    active_frac, avg_eps = [], []
    for r in range(N_ROUNDS):
        active = pop.active_users
        if not active:
            break
        participants = [u for u in active if rng.random() < p_participate]
        if participants:
            true_count, contributions = compute_round_conversions(participants, rng)
            is_top, _ = laplace_threshold_query(true_count, threshold, EPSILON, rng)
            for i, user in enumerate(participants):
                e, d = acc.charge(user=user, is_top=is_top,
                                  contributed=contributions[i], round_num=r)
                user.epsilon_spent += e
                user.delta_spent   += d
        pop.drop_depleted()
        eps_dist = pop.epsilon_remaining_distribution()
        active_frac.append(pop.active_fraction)
        avg_eps.append(float(np.mean(eps_dist)) if eps_dist else 0.0)
    return active_frac, avg_eps

## Section 1: Registration Cost vs. Budget

Standard IDP exhausts every user at exactly round `EPS_BUDGET/ε = 100`.  
TCT pays K₁·ε at registration then only small incremental charges on tops.  
TCT wins as long as K₁·ε < EPS_BUDGET, i.e. K₁ < 100.

Note: population E[rate] is irrelevant for Standard IDP (it charges everyone the same rate).

In [ ]:
DELTAS_TABLE = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6]

budget_threshold = EPS_BUDGET / EPSILON   # K₁ must be below this for TCT to win
idp_hl = int(EPS_BUDGET / EPSILON)        # Standard IDP half-life (all users exhaust together)

rows = []
for d in DELTAS_TABLE:
    acc = AdaptiveTCTAccountant(EPSILON, d)
    k1  = acc.compositions(1)
    reg = k1 * EPSILON
    wins = 'TCT ✓' if k1 < budget_threshold else 'DP  ✗'
    rows.append({
        'δ':         f'{d:.0e}',
        'K₁':        f'{k1:.1f}',
        'K₁·ε':      f'{reg:.3f}',
        'Budget':    f'{EPS_BUDGET:.1f}',
        'Affordable?': wins,
        'IDP hl':    str(idp_hl),
        'TCT adv':   f'>{N_ROUNDS - idp_hl}+ rounds',
    })

df = pd.DataFrame(rows).set_index('δ')
print(df.to_string())
print()
print(f'Budget threshold: K₁ < EPS_BUDGET/ε = {EPS_BUDGET}/{EPSILON} = {budget_threshold:.0f}')
print(f'Standard IDP half-life = {idp_hl} rounds (fixed, population-independent)')
print(f'All δ values satisfy K₁ < {budget_threshold:.0f} → TCT-ind always wins over Standard IDP')

## Section 2: δ Sweep

Fix population = **sparse** (α=0.3, β=3.0).  
Vary δ from 1e-6 to 1e-1 and measure the half-life gap.

Standard IDP half-life is fixed at ~100 regardless of δ.  
TCT-ind half-life increases as δ grows (smaller K₁ → cheaper registration → more budget remaining).

In [ ]:
DELTAS_SWEEP = [1e-6, 1e-5, 1e-4, 1e-3, 5e-3, 1e-2, 5e-2, 1e-1]

sparse_cfg = PopulationConfig(
    n_users=N_USERS, alpha=0.3, beta=3.0,
    epsilon_budget=EPS_BUDGET, delta_budget=1.0,
)
sparse_threshold = calibrate_threshold(sparse_cfg)

hl_tct_delta, hl_idp_delta = [], []

for d in DELTAS_SWEEP:
    tct_a, _ = run_tct_only(lambda d=d: AdaptiveTCTAccountant(EPSILON, d, independent=True),
                             sparse_cfg, sparse_threshold)
    idp_a, _ = run_tct_only(lambda: StandardDPAccountant(EPSILON),   # independent=False: all users pay
                             sparse_cfg, sparse_threshold)
    hl_tct_delta.append(half_life(tct_a))
    hl_idp_delta.append(half_life(idp_a))
    k1 = AdaptiveTCTAccountant(EPSILON, d).compositions(1)
    print(f'  δ={d:.0e}  K₁={k1:.1f}  K₁·ε={k1*EPSILON:.3f}  '
          f'TCT_hl={hl_tct_delta[-1]}  IDP_hl={hl_idp_delta[-1]}  gap={hl_tct_delta[-1]-hl_idp_delta[-1]:+d}')

gap_delta = [hl_tct_delta[i] - hl_idp_delta[i] for i in range(len(DELTAS_SWEEP))]
log_deltas = [-math.log10(d) for d in DELTAS_SWEEP]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(log_deltas, hl_tct_delta, 'o-', color='steelblue', lw=2, label='Adaptive TCT (ind)')
ax.plot(log_deltas, hl_idp_delta, 's--', color='gray',     lw=2, label='Standard IDP')
ax.set_xticks(log_deltas)
ax.set_xticklabels([f'{d:.0e}' for d in DELTAS_SWEEP], rotation=45, fontsize=8)
ax.set_xlabel('δ')
ax.set_ylabel('Half-life (TCT rounds)')
ax.set_title('Half-Life vs δ  (sparse population, E[rate]=0.091)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.bar(range(len(DELTAS_SWEEP)), gap_delta, color='steelblue', alpha=0.85)
ax.axhline(0, color='black', lw=1)
ax.set_xticks(range(len(DELTAS_SWEEP)))
ax.set_xticklabels([f'{d:.0e}' for d in DELTAS_SWEEP], rotation=45, fontsize=8)
ax.set_xlabel('δ')
ax.set_ylabel('TCT half-life − IDP half-life (rounds)')
ax.set_title('TCT-ind Advantage vs δ')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'δ sweep — sparse pop (α=0.3, β=3.0, E[rate]≈0.091), ε={EPSILON}  |  '
             f'Standard IDP hl≈{int(EPS_BUDGET/EPSILON)} (fixed)', fontsize=11)
plt.tight_layout()
plt.show()

## Section 3: Population Activity Sweep

Fix **δ=1e-3**. Vary population activity (α, β) to sweep E[conversion_rate].

Standard IDP half-life is **flat** — it charges all active users ε/round regardless of their individual conversion rate.  
TCT-ind half-life is also mostly flat, with slight variation from incremental charges on tops.  
This confirms that population E[rate] drives the *Standard DP-ind* cost but not *Standard IDP* cost.

In [ ]:
DELTA_FIXED = 1e-3
acc_ref = AdaptiveTCTAccountant(EPSILON, DELTA_FIXED)
print(f'Fixed δ={DELTA_FIXED:.0e}: K₁={acc_ref.compositions(1):.1f}, reg_cost={acc_ref.compositions(1)*EPSILON:.3f}ε')
print(f'Standard IDP half-life = {int(EPS_BUDGET/EPSILON)} rounds (population-independent)')
print()

pop_sweep = [
    (0.2, 5.0),   # E[rate] ≈ 0.038
    (0.3, 3.0),   # E[rate] ≈ 0.091
    (0.5, 3.0),   # E[rate] ≈ 0.143
    (0.5, 2.0),   # E[rate] = 0.200
    (0.5, 1.5),   # E[rate] ≈ 0.250
    (0.5, 1.0),   # E[rate] ≈ 0.333
    (1.0, 1.0),   # E[rate] = 0.500
]
e_rates = [a/(a+b) for a, b in pop_sweep]

hl_tct_pop, hl_idp_pop = [], []

for (a, b), e_rate in zip(pop_sweep, e_rates):
    cfg = PopulationConfig(n_users=N_USERS, alpha=a, beta=b,
                           epsilon_budget=EPS_BUDGET, delta_budget=1.0)
    thr = calibrate_threshold(cfg)
    tct_a, _ = run_tct_only(lambda: AdaptiveTCTAccountant(EPSILON, DELTA_FIXED, independent=True), cfg, thr)
    idp_a, _ = run_tct_only(lambda: StandardDPAccountant(EPSILON), cfg, thr)
    hl_tct_pop.append(half_life(tct_a))
    hl_idp_pop.append(half_life(idp_a))
    print(f'  E[rate]={e_rate:.3f} (α={a},β={b}):  TCT_hl={hl_tct_pop[-1]}  IDP_hl={hl_idp_pop[-1]}  gap={hl_tct_pop[-1]-hl_idp_pop[-1]:+d}')

gap_pop = [hl_tct_pop[i] - hl_idp_pop[i] for i in range(len(pop_sweep))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(e_rates, hl_tct_pop, 'o-', color='steelblue', lw=2, label='Adaptive TCT (ind)')
ax.plot(e_rates, hl_idp_pop, 's--', color='gray',     lw=2, label='Standard IDP')
ax.set_xlabel('E[conversion_rate]')
ax.set_ylabel('Half-life (TCT rounds)')
ax.set_title(f'Half-Life vs Population Activity  (δ={DELTA_FIXED:.0e})')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.bar(range(len(pop_sweep)), gap_pop, color='steelblue', alpha=0.85)
ax.axhline(0, color='black', lw=1)
ax.set_xticks(range(len(pop_sweep)))
ax.set_xticklabels([f'{r:.3f}' for r in e_rates], rotation=45)
ax.set_xlabel('E[conversion_rate]')
ax.set_ylabel('TCT half-life − IDP half-life (rounds)')
ax.set_title('TCT-ind Advantage vs Population Activity')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'Activity sweep — δ={DELTA_FIXED:.0e} (K₁={acc_ref.compositions(1):.1f}), ε={EPSILON}  |  '
             f'Standard IDP hl≈{int(EPS_BUDGET/EPSILON)} (flat — E[rate] irrelevant)', fontsize=11)
plt.tight_layout()
plt.show()

## Section 4: Phase Diagram — (δ, E[rate])

Full 2D grid showing where TCT-ind wins and where DP-ind wins.  
Color = TCT half-life minus DP half-life (green = TCT wins, red = DP wins).

In [ ]:
DELTAS_GRID  = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6]
POP_GRID = [
    (0.3, 3.0),   # E[rate] ≈ 0.091
    (0.5, 2.0),   # E[rate] = 0.200
    (0.5, 1.0),   # E[rate] ≈ 0.333
    (1.0, 1.0),   # E[rate] = 0.500
]
e_rates_grid = [a/(a+b) for a, b in POP_GRID]

print('Running phase diagram...', flush=True)
gap_matrix = np.zeros((len(DELTAS_GRID), len(POP_GRID)))

for j, (a, b) in enumerate(POP_GRID):
    cfg = PopulationConfig(n_users=N_USERS, alpha=a, beta=b,
                           epsilon_budget=EPS_BUDGET, delta_budget=1.0)
    thr = calibrate_threshold(cfg)
    # Standard IDP: charges all active users — run once per population (result is same for all δ)
    idp_a, _ = run_tct_only(lambda: StandardDPAccountant(EPSILON), cfg, thr)
    hl_idp = half_life(idp_a)
    for i, d in enumerate(DELTAS_GRID):
        tct_a, _ = run_tct_only(lambda d=d: AdaptiveTCTAccountant(EPSILON, d, independent=True), cfg, thr)
        gap_matrix[i, j] = half_life(tct_a) - hl_idp
        print(f'  δ={d:.0e} E[rate]={a/(a+b):.3f}: gap={gap_matrix[i,j]:+.0f}', flush=True)

fig, ax = plt.subplots(figsize=(9, 6))
vmax = max(abs(gap_matrix.min()), abs(gap_matrix.max()))
im = ax.imshow(gap_matrix, cmap='RdYlGn', vmin=-vmax, vmax=vmax, aspect='auto')
plt.colorbar(im, ax=ax, label='TCT half-life − IDP half-life (rounds)')

ax.set_xticks(range(len(POP_GRID)))
ax.set_xticklabels([f'{r:.3f}' for r in e_rates_grid])
ax.set_yticks(range(len(DELTAS_GRID)))
ax.set_yticklabels([f'{d:.0e}' for d in DELTAS_GRID])
ax.set_xlabel('E[conversion_rate]')
ax.set_ylabel('δ')
ax.set_title(f'Phase diagram: TCT-ind vs Standard IDP half-life gap\n'
             f'ε={EPSILON}, N={N_USERS}, budget={EPS_BUDGET}  |  green=TCT wins  '
             f'(Standard IDP hl≈{int(EPS_BUDGET/EPSILON)}, population-independent)')

for i in range(len(DELTAS_GRID)):
    for j in range(len(POP_GRID)):
        ax.text(j, i, f'{gap_matrix[i,j]:+.0f}', ha='center', va='center',
                fontsize=10, color='black', fontweight='bold')

plt.tight_layout()
plt.show()

## Section 5: Winning Regime — Full Simulation

Pick parameters from the green region of the phase diagram and run the full
three-way comparison (TCT-ind vs DP-ind vs DP) with interleaved non-TCT queries.

**Chosen:** δ=1e-2, moderate population (E[rate]=0.2) — TCT should win clearly.

In [ ]:
# ── Winning regime parameters ─────────────────────────────────────────────────
DELTA_WIN       = 1e-2
POP_ALPHA_WIN   = 0.5
POP_BETA_WIN    = 2.0    # E[rate] = 0.2

# ── Non-TCT query model (reuses P_PARTICIPATE from shared params) ─────────────
LAMBDA_NONTCT  = 3.0
EPS_NONTCT_MIN = 0.005
EPS_NONTCT_MAX = 0.02

win_cfg = PopulationConfig(
    n_users=N_USERS, alpha=POP_ALPHA_WIN, beta=POP_BETA_WIN,
    epsilon_budget=EPS_BUDGET, delta_budget=1.0,
)
win_threshold = calibrate_threshold(win_cfg)   # uses P_PARTICIPATE

acc_win = AdaptiveTCTAccountant(EPSILON, DELTA_WIN)
print(f'Winning regime: δ={DELTA_WIN:.0e}, E[rate]={POP_ALPHA_WIN/(POP_ALPHA_WIN+POP_BETA_WIN):.3f}')
print(f'K₁={acc_win.compositions(1):.1f}, reg_cost={acc_win.compositions(1)*EPSILON:.3f}ε')
print(f'Standard IDP expected exhaustion ≈ {int(EPS_BUDGET/(P_PARTICIPATE*EPSILON))} TCT rounds '
      f'(partial participation p={P_PARTICIPATE})')
print()


def run_full(accountant_factory, independent_nontct, seed=SEED):
    """Full simulation: both TCT and non-TCT queries use partial participation.

    TCT rounds: P_PARTICIPATE fraction of active users is sampled per round.
      - TCT accountant charges only sampled users; only contributors to tops pay incremental.
      - Standard IDP charges all sampled users ε.
    Non-TCT rounds: Poisson(λ) queries, each with P_PARTICIPATE participation and
      Uniform(eps_min, eps_max) cost.
    """
    rng = np.random.default_rng(seed)
    pop = generate_synthetic_population(win_cfg, rng=np.random.default_rng(seed))
    acc = accountant_factory()
    contrib_counts = defaultdict(int)
    active_frac, avg_eps = [], []

    for r in range(N_ROUNDS):
        active = pop.active_users
        if not active:
            break

        # ── TCT round: sample participants ────────────────────────────────────
        participants = [u for u in active if rng.random() < P_PARTICIPATE]
        if participants:
            true_count, contributions = compute_round_conversions(participants, rng)
            is_top, _ = laplace_threshold_query(true_count, win_threshold, EPSILON, rng)
            for i, user in enumerate(participants):
                if contributions[i]:
                    contrib_counts[user.user_id] += 1
                e, d = acc.charge(user=user, is_top=is_top,
                                  contributed=contributions[i], round_num=r)
                user.epsilon_spent += e
                user.delta_spent   += d

        # ── Non-TCT queries: Poisson count, uniform cost, per-user participation ──
        n_queries = rng.poisson(LAMBDA_NONTCT)
        for _ in range(n_queries):
            cost = rng.uniform(EPS_NONTCT_MIN, EPS_NONTCT_MAX)
            for user in pop.active_users:
                participates = rng.random() < P_PARTICIPATE
                if not independent_nontct or participates:
                    user.epsilon_spent += cost

        pop.drop_depleted()
        eps_dist = pop.epsilon_remaining_distribution()
        active_frac.append(pop.active_fraction)
        avg_eps.append(float(np.mean(eps_dist)) if eps_dist else 0.0)

    final_eps = {u.user_id: max(u.epsilon_remaining, 0.0) for u in pop.all_users}
    return active_frac, avg_eps, dict(contrib_counts), final_eps


print('Running full simulations...')
tct_a, tct_e, tct_c, tct_f = run_full(
    lambda: AdaptiveTCTAccountant(EPSILON, DELTA_WIN, independent=True),
    independent_nontct=True)
idp_a, idp_e, idp_c, idp_f = run_full(
    lambda: StandardDPAccountant(EPSILON),   # independent=False: all sampled users pay
    independent_nontct=False)
print('Done.')

hl_tct = half_life(tct_a)
hl_idp = half_life(idp_a)

def pad(c, n): return list(c) + [c[-1]] * (n - len(c))
n = max(len(tct_a), len(idp_a)) + 1
rnds = list(range(n))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(rnds, pad(tct_a, n), color='steelblue', lw=2,
        label=f'Adaptive TCT (ind)  hl={hl_tct}')
ax.plot(rnds, pad(idp_a, n), color='gray',      lw=2, linestyle='--',
        label=f'Standard IDP        hl={hl_idp}')
ax.axhline(0.5, color='silver', lw=1, linestyle=':')
ax.set_xlabel('TCT Round')
ax.set_ylabel('Fraction of users active')
ax.set_title('User Retention — Winning Regime')
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(rnds, pad(tct_e, n), color='steelblue', lw=2, label='Adaptive TCT (ind)')
ax.plot(rnds, pad(idp_e, n), color='gray',      lw=2, linestyle='--', label='Standard IDP')
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('TCT Round')
ax.set_ylabel('Avg ε remaining (active users)')
ax.set_title('Average Budget Remaining')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle(
    f'δ={DELTA_WIN:.0e}, α={POP_ALPHA_WIN}, β={POP_BETA_WIN} (E[rate]=0.2)  |  '
    f'p_participate={P_PARTICIPATE}  |  non-TCT: Poisson(λ={LAMBDA_NONTCT})',
    fontsize=10
)
plt.tight_layout()
plt.show()

print(f'Half-life: TCT-ind={hl_tct},  Standard IDP={hl_idp}')
print(f'Advantage: {hl_tct - hl_idp:+d} rounds')

## Section 6: Hit Rate (Threshold Level) Sweep

`TARGET_HIT_RATE` sets how often a round crosses threshold T.
`q = 1/(e^ε+1) ≈ 0.4975` is fixed by ε alone — it is NOT the hit rate.

**Prediction:**
- **Standard IDP half-life is flat** — it charges all active users ε/round regardless of whether a top occurred or where the threshold sits.
- **TCT-ind half-life decreases as hit rate rises** — more tops → more incremental charges on top of the fixed registration cost K₁·ε. At hit rate → q ≈ 0.4975, TCT-ind converges toward Standard IDP behaviour.

Run with δ=1e-2, moderate pop (E[rate]=0.2) from Section 5.

In [ ]:
from simulation.calibration import set_threshold_for_hit_rate

HIT_RATES = [0.005, 0.01, 0.02, 0.05, 0.10, 0.20, 0.35, 0.50]

acc_ref2 = AdaptiveTCTAccountant(EPSILON, DELTA_WIN)
q_val    = acc_ref2.q
k1_val   = acc_ref2.compositions(1)
print(f'q={q_val:.4f}  K₁={k1_val:.1f}  reg_cost={k1_val*EPSILON:.3f}ε  (δ={DELTA_WIN:.0e}, ε={EPSILON})')
print(f'Standard IDP hl = {int(EPS_BUDGET/EPSILON)} (flat across all hit rates)')
print()

hl_tct_hr, hl_idp_hr = [], []

for hr in HIT_RATES:
    cfg = PopulationConfig(
        n_users=N_USERS, alpha=POP_ALPHA_WIN, beta=POP_BETA_WIN,
        epsilon_budget=EPS_BUDGET, delta_budget=1.0,
    )
    pop_cal = generate_synthetic_population(cfg, rng=np.random.default_rng(SEED))
    thr, emp_hr = set_threshold_for_hit_rate(
        pop_cal, EPSILON, hr, n_pilot=2000, rng=np.random.default_rng(SEED + 7))

    tct_a, _ = run_tct_only(
        lambda: AdaptiveTCTAccountant(EPSILON, DELTA_WIN, independent=True), cfg, thr)
    idp_a, _ = run_tct_only(
        lambda: StandardDPAccountant(EPSILON), cfg, thr)   # independent=False: all users pay
    hl_tct_hr.append(half_life(tct_a))
    hl_idp_hr.append(half_life(idp_a))
    print(f'  hit_rate={hr:.3f} (emp={emp_hr:.3f}, T={thr:.1f}):  '
          f'TCT_hl={hl_tct_hr[-1]}  IDP_hl={hl_idp_hr[-1]}  gap={hl_tct_hr[-1]-hl_idp_hr[-1]:+d}')

gap_hr = [hl_tct_hr[i] - hl_idp_hr[i] for i in range(len(HIT_RATES))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(HIT_RATES, hl_tct_hr, 'o-', color='steelblue', lw=2, label='Adaptive TCT (ind)')
ax.plot(HIT_RATES, hl_idp_hr, 's--', color='gray',     lw=2, label='Standard IDP')
ax.axvline(q_val, color='red', lw=1.2, linestyle=':', label=f'q={q_val:.4f}  (TCT degenerates)')
ax.set_xlabel('Target hit rate (fraction of rounds with a top)')
ax.set_ylabel('Half-life (TCT rounds)')
ax.set_title('Half-Life vs Threshold Level')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.bar(range(len(HIT_RATES)), gap_hr, color='steelblue', alpha=0.85)
ax.axhline(0, color='black', lw=1)
ax.set_xticks(range(len(HIT_RATES)))
ax.set_xticklabels([f'{r:.3f}' for r in HIT_RATES], rotation=45)
ax.set_xlabel('Target hit rate')
ax.set_ylabel('TCT half-life − IDP half-life (rounds)')
ax.set_title('TCT-ind Advantage vs Threshold Level')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(
    f'Hit rate sweep — δ={DELTA_WIN:.0e} (K₁={k1_val:.1f}), '
    f'α={POP_ALPHA_WIN} β={POP_BETA_WIN} (E[rate]=0.2), ε={EPSILON}\n'
    f'Standard IDP hl≈{int(EPS_BUDGET/EPSILON)} (flat); TCT-ind degrades as hit_rate → q={q_val:.4f}',
    fontsize=10
)
plt.tight_layout()
plt.show()